# 01. Normalización reproducible de las fuentes

Transformamos las fuentes certificadas en tablas tipadas y documentadas para
el análisis posterior. Cada registro conserva su origen y las tablas
normalizadas se publican junto con una cuarentena inspeccionable. Los faltantes
no se eliminan ni se sustituyen globalmente por cero. La aceptación depende
del uso previsto: una coordenada ausente impide una operación espacial, mientras
que un teléfono ausente no invalida una unidad económica.

In [1]:
import json
import sys
from pathlib import Path
import pandas as pd
import geopandas as gpd
import pyarrow.parquet as pq
from IPython.display import display, Markdown

BASE = Path.cwd().resolve()
PROJECT = next(p / "4_proyecto" for p in [BASE, *BASE.parents] if (p / "4_proyecto/V3/data").is_dir())
V3 = PROJECT / "V3"
sys.path.insert(0, str(PROJECT))
from V3.src.etl_contract import build_manifest, verify_manifest, provenance, get_processed_root, get_quarantine_root
from V3.src.enterprise_etl import run_etl, verify_outputs
print("Directorio del proyecto:", V3)

assert (V3 / "audits/00_manifest.json").is_file(), "Ejecutar primero el cuaderno00."
manifest = verify_manifest(V3)

Directorio del proyecto: /Users/oscarmartinez/Desktop/Caso_De_negocio_ITAM/V3


## Ejecución y publicación de una generación

Aplicamos reglas específicas por fuente, normalizamos texto a UTF-8/NFC y
preservamos identificadores como cadenas cuando sus ceros iniciales tienen
significado. Los conteos utilizan enteros nulables y los pesos de eventos
utilizan valores fraccionarios. La deduplicación utiliza fechas de actualización
cuando están disponibles; sin ellas, el desempate es determinista por hash
y fila, sin afirmar que la fecha de registro represente una actualización.

La publicación utiliza staging, bloqueo exclusivo y comprobaciones de
conservación. La auditoría final actúa como marcador de compromiso: los
consumidores rechazan generaciones incompletas. Las tablas de esta versión
son insumos preparados; no constituyen todavía una matriz PCA ni una
validación causal, temporal o predictiva.

In [2]:
audit = run_etl(V3)
processed = get_processed_root(V3)
quarantine = get_quarantine_root(V3)
display(pd.Series(audit["stations"], name="estaciones").to_frame())
display(pd.Series(audit["agebs"], name="agebs").to_frame())
display(pd.Series(audit["trip_rows"], name="viajes").to_frame())
auxiliary_quality = pd.DataFrame(audit["auxiliary"]["tables"]).T
display(auxiliary_quality)
display(Markdown(f"**Resultados.** La generación conserva {audit['stations']['accepted_rows']:,} "
                 f"estaciones y {audit['agebs']['accepted_rows']:,} AGEBs. En viajes, "
                 f"{audit['trip_rows']['accepted_rows']:,} filas cumplen el contrato y "
                 f"{audit['trip_rows']['quarantine_rows']:,} se conservan en cuarentena."))

data/raw/ecobici_trips/2025-01.csv: 1809775 rows; quarantine 143


data/raw/ecobici_trips/2025-02.csv: 1745620 rows; quarantine 33


data/raw/ecobici_trips/2025-03.csv: 1832587 rows; quarantine 38


data/raw/ecobici_trips/2025-04.csv: 1727987 rows; quarantine 726


data/raw/ecobici_trips/2025-05.csv: 1778594 rows; quarantine 49


...

,estaciones
input_rows,677
accepted_rows,677
quarantine_rows,0
superseded_duplicates,0
feed_last_updated,2026-09-06T07:52:55+00:00
snapshot_is_not_2025_active_station_evidence,True
compound_physical_stations,12


,agebs
input_rows,2452
accepted_rows,2452
quarantine_rows,0
superseded_duplicates,0
rural_nan_rows,22
population_missing_rows,22
crs_respects_per_feature_declared_epsg6365,True


,viajes
input_rows,20134900
accepted_rows,20133540
quarantine_rows,1360
valid_retiros,20134368
valid_arribos,20134527


,input_rows,accepted_rows,quarantine_rows,superseded_duplicates,null_counts,reason_occurrences,ageb_view_rows,geographic_levels,resource_records,not_an_observation_table
censo_2020,68941,68941,0,0,"{'ENTIDAD': 0, 'NOM_ENT': 0, 'MUN': 0, 'NOM_MU...",{},2433,"{'manzana': 66456, 'ageb': 2433, 'localidad': ...",NaN,NaN
denue,462732,462732,0,0,"{'id': 0, 'clee': 0, 'nom_estab': 212, 'raz_so...",{},NaN,NaN,NaN,NaN
gtfs_agency,10,10,0,0,"{'agency_id': 0, 'agency_name': 0, 'agency_url...",{},NaN,NaN,NaN,NaN
gtfs_calendar,13,13,0,0,"{'service_id': 0, 'monday': 0, 'tuesday': 0, '...",{},NaN,NaN,NaN,NaN
gtfs_stops,11362,11362,0,0,"{'stop_id': 0, 'stop_name': 0, 'stop_lat': 0, ...",{},NaN,NaN,NaN,NaN
gtfs_routes,301,300,1,0,"{'route_id': 0, 'agency_id': 0, 'route_short_n...",{'unknown_agency': 1},NaN,NaN,NaN,NaN
gtfs_shapes,127135,127135,0,0,"{'shape_id': 0, 'shape_pt_lat': 0, 'shape_pt_l...",{},NaN,NaN,NaN,NaN
gtfs_trips,1205,1204,1,0,"{'route_id': 0, 'service_id': 0, 'trip_id': 0,...",{'unknown_route': 1},NaN,NaN,NaN,NaN
gtfs_stop_times,42789,42789,0,0,"{'trip_id': 0, 'timepoint': 0, 'stop_id': 0, '...",{},NaN,NaN,NaN,NaN
gtfs_frequencies,1584,1583,1,0,"{'trip_id': 0, 'start_time': 0, 'end_time': 0,...",{'unknown_trip': 1},NaN,NaN,NaN,NaN


**Resultados.** La generación conserva 677 estaciones y 2,452 AGEBs. En viajes, 20,133,540 filas cumplen el contrato y 1,360 se conservan en cuarentena.

## Identidad de estaciones, actividad y uso extremo

Resolvemos los códigos dobles del catálogo mediante una tabla de alias. Los
componentes de `158-159`, por ejemplo, apuntan a una sola estación física.
Esto evita perder eventos por códigos secundarios o crear estaciones nuevas
sin respaldo en el catálogo.

La actividad diaria es `(retiros + arribos) / (2 × 365)`. Marcamos las
estaciones por encima de `Q3 + 1.5 × IQR`, sin excluirlas. El criterio IQR
no define el Golden Cluster: esa definición por P85 corresponde al EDA.
El snapshot GBFS de 2026 no acredita por sí solo operación de cada estación
durante todo 2025.

In [3]:
stations = pd.read_parquet(processed / "estaciones.parquet")
aliases = pd.read_parquet(processed / "station_aliases.parquet")
activity = pd.read_parquet(processed / "actividad_estaciones.parquet")
display(stations[["ce_id", "name", "_updated_at", "catalog_snapshot_outside_2025"]].head())
display(aliases.head())
display(activity[["ce_id", "viajes_diarios_prom", "is_outlier_high"]].head())
assert activity.dias_calendario.eq(365).all()
assert len(activity) == len(stations)
display(Markdown(f"**Resultados.** {len(aliases):,} códigos se resuelven en {len(stations):,} "
                 f"estaciones físicas. Conservamos las {int(activity.is_outlier_high.sum()):,} "
                 "estaciones señaladas por IQR."))

,ce_id,name,_updated_at,catalog_snapshot_outside_2025
0,001,CE-001 Río Sena-Río Balsas,2026-09-06 07:52:55+00:00,True
1,002,CE-002 Río Guadalquivir - Río Nazas,2026-09-06 07:52:55+00:00,True
2,003,CE-003 Reforma - Insurgentes,2026-09-06 07:52:55+00:00,True
3,004,CE-004 Río Nilo - Río Panuco,2026-09-06 07:52:55+00:00,True
4,005,CE-005 Río Pánuco Río Tiber,2026-09-06 07:52:55+00:00,True


,station_code,ce_id,_source_file,_processed_at
0,001,001,data/raw/ecobici_stations/station_information....,2026-10-11 23:49:59+00:00
1,002,002,data/raw/ecobici_stations/station_information....,2026-10-11 23:49:59+00:00
2,003,003,data/raw/ecobici_stations/station_information....,2026-10-11 23:49:59+00:00
3,004,004,data/raw/ecobici_stations/station_information....,2026-10-11 23:49:59+00:00
4,005,005,data/raw/ecobici_stations/station_information....,2026-10-11 23:49:59+00:00


,ce_id,viajes_diarios_prom,is_outlier_high
0,001,237.290411,True
1,002,98.186301,False
2,003,107.708219,False
3,004,137.09863,False
4,005,139.136986,False


**Resultados.** 689 códigos se resuelven en 677 estaciones físicas. Conservamos las 36 estaciones señaladas por IQR.

## Geometrías AGEB y tabulados del Censo

Conservamos las AGEBs rurales sin población disponible y mantenemos su
densidad como faltante. Respetamos el CRS declarado de las geometrías y
calculamos las áreas en EPSG:32614. La cuarentena conserva las geometrías que
no cumplen las comprobaciones de forma, finitud y tipo.

En el Censo distinguimos totales de entidad, municipio, localidad, AGEB y
manzana. Conservamos todas las filas y publicamos una vista de totales AGEB
para evitar sumar repetidamente niveles de agregación. Los asteriscos de
supresión estadística se representan como nulos con indicadores explícitos.
La ausencia de un dato suprimido no se interpreta como cero.

In [4]:
agebs = gpd.read_parquet(processed / "agebs.parquet")
census_ageb = pd.read_parquet(processed / "censo_ageb_2020.parquet")
rural = agebs.loc[agebs.is_rural_nan.fillna(False)]
display(rural[["cvegeo", "pob_total", "densidad_pob_km2", "is_rural_nan"]])
display(census_ageb[["cvegeo", "POBTOT", "has_suppressed_values"]].head())
assert len(rural) == 22 and rural.pob_total.isna().all() and rural.densidad_pob_km2.isna().all()
assert census_ageb.cvegeo.is_unique
display(Markdown(f"**Resultados.** Preservamos {len(rural)} AGEBs rurales sin población "
                 f"y publicamos {len(census_ageb):,} totales AGEB del Censo. "
                 "La conservación de nulos evita fabricar densidades o conteos."))

,cvegeo,pob_total,densidad_pob_km2,is_rural_nan
290,090040157,<NA>,<NA>,True
291,090040335,<NA>,<NA>,True
1217,090080368,<NA>,<NA>,True
1256,090090100,<NA>,<NA>,True
1257,090090115,<NA>,<NA>,True
1262,090090524,<NA>,<NA>,True
1263,090090539,<NA>,<NA>,True
1463,090101326,<NA>,<NA>,True
1573,090110471,<NA>,<NA>,True
1574,090110838,<NA>,<NA>,True


,cvegeo,POBTOT,has_suppressed_values
0,0900200010010,3183,True
1,0900200010025,5593,True
2,090020001003A,4235,True
3,0900200010044,4768,True
4,0900200010097,2176,True


**Resultados.** Preservamos 22 AGEBs rurales sin población y publicamos 2,433 totales AGEB del Censo. La conservación de nulos evita fabricar densidades o conteos.

### Hallazgos de correspondencia territorial

En la ejecución registrada en `audits/01_findings_v3.json` conservamos las
68,941 filas censales y obtuvimos 2,433 totales AGEB, sin mezclar esos totales
con sus manzanas. Las 2,430 geometrías urbanas tienen correspondencia censal;
tres claves censales no aparecen en la capa geométrica disponible. Conservamos
esas claves sin fabricar polígonos. Las diferencias de marco territorial
requieren revisión explícita antes de construir atributos espaciales.

Estos resultados corresponden a la generación de fuentes completas registrada
en esa auditoría. La aceptación estructural no implica ausencia de supresión
estadística ni sustituye una comprobación sustantiva de cada variable.

## Unidades económicas DENUE

Validamos identificadores, clasificación SCIAN y coordenadas, conservando
atributos opcionales como teléfono, correo y razón social cuando son nulos.
Señalamos claves AGEB sin correspondencia con el marco geográfico sin
atribuirles automáticamente una localización administrativa nueva. Las
categorías de personal ocupado permanecen como intervalos; una estimación
puntual de empleo requerirá una decisión explícita en feature engineering.

Los metadatos identifican esta edición como DENUE 2026. La conservamos con
esa advertencia y no afirmamos que represente establecimientos existentes en
2025. La fecha de alta tampoco se utiliza como fecha de actualización.

In [5]:
denue_quality = audit["auxiliary"]["tables"]["denue"]
display(pd.Series(denue_quality, name="calidad_denue").to_frame())
batch = next(pq.ParquetFile(processed / "denue.parquet").iter_batches(batch_size=5), None)
display(batch.to_pandas() if batch is not None else pd.DataFrame())
display(Markdown(f"**Resultados.** DENUE: {denue_quality['accepted_rows']:,} registros aceptados "
                 f"y {denue_quality['quarantine_rows']:,} conservados en cuarentena. "
                 "El informe distingue nulos por atributo y problemas de identidad o coordenadas."))

,calidad_denue
input_rows,462732
accepted_rows,462732
quarantine_rows,0
superseded_duplicates,0
null_counts,"{'id': 0, 'clee': 0, 'nom_estab': 212, 'raz_so..."
reason_occurrences,{}


,id,clee,nom_estab,raz_social,codigo_act,nombre_act,per_ocu,tipo_vial,nom_vial,tipo_v_e_1,...,coordinates_zero_pair,cvegeo,ageb_key_unmatched,snapshot_year,snapshot_outside_study_year,fecha_alta_parsed,registration_date_invalid,registered_after_study_year,employment_band_unknown,_dedup_key
0,12299604,09017112519000021000000000U5,ACUARIO AQUAVIRTUAL,<NA>,112519,Otra acuicultura,0 a 5 personas,CALLE,PROCURADURIA GENERAL DE JUSTICIA,CALLE,...,False,0901700011204,False,2026,True,2026-04-01,False,True,False,"[""12299604""]"
1,11845788,09013112515000091000000000U5,AJOLOTARIO APATLACO,<NA>,112519,Otra acuicultura,0 a 5 personas,OTRO(ESPECIFIQUE),NINGUNO,CALLE,...,False,0901301691281,True,2026,True,2025-05-01,False,False,False,"[""11845788""]"
2,11845785,09013112515000071000000000U7,AJOLOTARIO CARRIZAL,<NA>,112519,Otra acuicultura,0 a 5 personas,OTRO(ESPECIFIQUE),NINGUNO,CALLE,...,False,0901301471281,True,2026,True,2025-05-01,False,False,False,"[""11845785""]"
3,11845786,09013112515000082000000000U4,AJOLOTARIO CHINAMPA ONKALI,ONKALI,112519,Otra acuicultura,6 a 10 personas,OTRO(ESPECIFIQUE),CANAL,CALLE,...,False,0901300011099,False,2026,True,2025-05-01,False,False,False,"[""11845786""]"
4,11845789,09013112515000102000000000U0,AJOLOTARIO TLAZOCAMA TONANA,<NA>,112519,Otra acuicultura,6 a 10 personas,OTRO(ESPECIFIQUE),CANAL,CALLEJON,...,False,0901300011099,False,2026,True,2025-05-01,False,False,False,"[""11845789""]"


**Resultados.** DENUE: 462,732 registros aceptados y 0 conservados en cuarentena. El informe distingue nulos por atributo y problemas de identidad o coordenadas.

### Hallazgos de DENUE

La ejecución registrada conserva 462,732 establecimientos sin rechazos por
identidad, clasificación SCIAN o coordenadas. Sin embargo, 1,498 registros
tienen claves AGEB sin correspondencia con la capa geográfica disponible.
El indicador `ageb_key_unmatched` permite identificarlos sin excluirlos ni
asignarlos automáticamente a una AGEB.

La edición es 2026 y 9,627 registros tienen fecha de alta posterior a 2025.
No interpretamos la fecha de alta de los restantes establecimientos como
prueba de que seguían operando en 2025. La utilización del directorio como
contexto urbano requiere conservar esta diferencia temporal.

## GTFS y consistencia de relaciones

Normalizamos agencias, rutas, servicios, paradas, viajes, secuencias y
frecuencias, comprobando las referencias entre tablas. Los horarios GTFS
pueden superar las 24 horas, por lo que se convierten a segundos sin tratarlos
como errores de reloj. Los campos opcionales permanecen nulos y los valores
predeterminados se aplican únicamente cuando están definidos por el formato.
El calendario conserva su vigencia real; los servicios de 2026 no se
presentan como evidencia de servicio en 2025.

In [6]:
gtfs_quality = {name: report for name, report in audit["auxiliary"]["tables"].items()
                if name.startswith("gtfs_")}
display(pd.DataFrame(gtfs_quality).T)
display(pd.read_parquet(processed / "gtfs_calendar.parquet").head())
display(pd.read_parquet(processed / "gtfs_stops.parquet").head())
display(Markdown("**Resultados.** Las particiones publicadas identifican los rechazos por "
                 "clave, coordenada, horario o referencia. La vigencia temporal permanece explícita."))

,input_rows,accepted_rows,quarantine_rows,superseded_duplicates,reason_occurrences,null_counts
gtfs_agency,10,10,0,0,{},"{'agency_id': 0, 'agency_name': 0, 'agency_url..."
gtfs_calendar,13,13,0,0,{},"{'service_id': 0, 'monday': 0, 'tuesday': 0, '..."
gtfs_stops,11362,11362,0,0,{},"{'stop_id': 0, 'stop_name': 0, 'stop_lat': 0, ..."
gtfs_routes,301,300,1,0,{'unknown_agency': 1},"{'route_id': 0, 'agency_id': 0, 'route_short_n..."
gtfs_shapes,127135,127135,0,0,{},"{'shape_id': 0, 'shape_pt_lat': 0, 'shape_pt_l..."
gtfs_trips,1205,1204,1,0,{'unknown_route': 1},"{'route_id': 0, 'service_id': 0, 'trip_id': 0,..."
gtfs_stop_times,42789,42789,0,0,{},"{'trip_id': 0, 'timepoint': 0, 'stop_id': 0, '..."
gtfs_frequencies,1584,1583,1,0,{'unknown_trip': 1},"{'trip_id': 0, 'start_time': 0, 'end_time': 0,..."


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date,...,_processed_at,_source_row,_raw_record,_record_sha256,_updated_at,_quarantine_reason,start_date_parsed,end_date_parsed,has_2025_service,_dedup_key
0,B_0,1,1,1,1,1,1,1,20241201,20251231,...,2026-10-11 23:49:59+00:00,8,"{""end_date"":""20251231"",""friday"":""1"",""monday"":""...",da8233c5c27dc7df577c4408b2e66a6ee6a8e5d04564f5...,NaT,,2024-12-01,2025-12-31,True,"[""B_0""]"
1,B_1,1,1,1,1,1,0,0,20241201,20251231,...,2026-10-11 23:49:59+00:00,11,"{""end_date"":""20251231"",""friday"":""1"",""monday"":""...",256181a7aa4a7b1af64ff65c2abba82c26b457c57ad3b4...,NaT,,2024-12-01,2025-12-31,True,"[""B_1""]"
2,B_2,0,0,0,0,0,1,0,20241201,20251231,...,2026-10-11 23:49:59+00:00,10,"{""end_date"":""20251231"",""friday"":""0"",""monday"":""...",24bf217ad3aee2d1dd8418b95e78f6af277b3b53baba23...,NaT,,2024-12-01,2025-12-31,True,"[""B_2""]"
3,B_3,0,0,0,0,0,0,1,20241201,20251231,...,2026-10-11 23:49:59+00:00,9,"{""end_date"":""20251231"",""friday"":""0"",""monday"":""...",7e01a9934cf702ee29a33bc34657d9a87cbc9a8e99220d...,NaT,,2024-12-01,2025-12-31,True,"[""B_3""]"
4,B_4,1,1,1,1,1,1,0,20241201,20251231,...,2026-10-11 23:49:59+00:00,7,"{""end_date"":""20251231"",""friday"":""1"",""monday"":""...",a22290edef6ecef7da69eec76029bf040772670c0508a4...,NaT,,2024-12-01,2025-12-31,True,"[""B_4""]"


,stop_id,stop_name,stop_lat,stop_lon,zone_id,wheelchair_boarding,_source_file,_processed_at,_source_row,_raw_record,_record_sha256,_updated_at,_quarantine_reason,location_type,has_valid_coordinates,coordinates_zero_pair,_dedup_key
0,B_0100C100-CNLCHALCO,Base Periférico Canal de Chalco,19.30387,-99.08736,0100C100-CNLCHALCO,2,data/raw/gtfs/gtfs.zip::stops.txt,2026-10-11 23:49:59+00:00,1365,"{""stop_id"":""B_0100C100-CNLCHALCO"",""stop_lat"":""...",d88af5680e3d40afa86e1614f45b59c115fceae9f1cd8e...,NaT,,0,True,False,"[""B_0100C100-CNLCHALCO""]"
1,B_0100C100-PERIF10OTE,Periférico - 10 Oriente,19.30186,-99.16858,0100C100-PERIF10OTE,2,data/raw/gtfs/gtfs.zip::stops.txt,2026-10-11 23:49:59+00:00,1380,"{""stop_id"":""B_0100C100-PERIF10OTE"",""stop_lat"":...",83b206a0aeeb97cba4756233497254ec842434084a632e...,NaT,,0,True,False,"[""B_0100C100-PERIF10OTE""]"
2,B_0100C100-PERIF11ABRIL,Periférico - 11 de Abril,19.39528,-99.18918,0100C100-PERIF11ABRIL,2,data/raw/gtfs/gtfs.zip::stops.txt,2026-10-11 23:49:59+00:00,1418,"{""stop_id"":""B_0100C100-PERIF11ABRIL"",""stop_lat...",ab44ae7e881928d7568589b443249b8ff89a6c5afa991a...,NaT,,0,True,False,"[""B_0100C100-PERIF11ABRIL""]"
3,B_0100C100-PERIFAGUILAS,Periférico - Las Aguilas,19.3583,-99.19509,0100C100-PERIFAGUILAS,2,data/raw/gtfs/gtfs.zip::stops.txt,2026-10-11 23:49:59+00:00,1407,"{""stop_id"":""B_0100C100-PERIFAGUILAS"",""stop_lat...",cea9a848c563eb42aa301ff742487e06e49ffe5a24380b...,NaT,,0,True,False,"[""B_0100C100-PERIFAGUILAS""]"
4,B_0100C100-PERIFALENCAS,Periférico - Alencastre,19.42262,-99.19904,0100C100-PERIFALENCAS,2,data/raw/gtfs/gtfs.zip::stops.txt,2026-10-11 23:49:59+00:00,1424,"{""stop_id"":""B_0100C100-PERIFALENCAS"",""stop_lat...",edd16b94d55edcfc05c8a47c85c6984a484fb12d4b4cf9...,NaT,,0,True,False,"[""B_0100C100-PERIFALENCAS""]"


**Resultados.** Las particiones publicadas identifican los rechazos por clave, coordenada, horario o referencia. La vigencia temporal permanece explícita.

### Hallazgos de integridad referencial GTFS

Identificamos una ruta, un viaje y una frecuencia que no cumplen las
referencias entre tablas. La ruta `TR13` declara la agencia `SEMOVI`, que
no está presente en `agency.txt`; su viaje y frecuencia quedan también
fuera de la partición válida. Conservamos los tres registros con sus
motivos, sin inventar una equivalencia de agencias. El calendario incluye
doce servicios con intervalos que intersectan 2025 y un servicio de 2026.

## Infraestructura ciclista y fuentes de incidentes

Validamos las geometrías y conservamos por separado cada capa de ciclovías.
La capa total no se concatena con sus componentes, porque hacerlo duplicaría
longitudes. El GeoJSON alternativo y el shapefile mantienen sus identidades
de fuente; tampoco se suman como infraestructura independiente.

En C5 y accidentes delimitamos una cohorte analítica de 2025. Los registros
históricos quedan en cuarentena como fuera del universo, no necesariamente
como datos corruptos. En SSC conservamos registros de víctimas y señalamos
posibles repeticiones, sin convertir coincidencias de fecha y posición en
eventos únicos demostrados. Las fuentes viales no se concatenan como si sus
observaciones fueran independientes.

In [7]:
display(pd.read_parquet(processed / "ciclovias_capas.parquet"))
incident_quality = {name: report for name, report in audit["auxiliary"]["tables"].items()
                    if name.startswith(("incidentes_", "accidentes_", "ssc_"))}
display(pd.DataFrame(incident_quality).T)
display(pd.read_parquet(processed / "ssc_victimas_2025.parquet").head())
display(Markdown("**Resultados.** Publicamos capas y tablas con su partición de aceptación "
                 "y cuarentena. Los indicadores de uso, vigencia y repetición se conservan "
                 "para seleccionar fuentes y unidades de análisis en las etapas siguientes."))

,artifact,layer,is_total_layer,not_additive_with_other_layers,_source_file,_processed_at
0,ciclovias_capa_00.parquet,Carril Bus Bici,False,True,data/raw/urban_environment/ciclovias_semovi_20...,2026-10-11 23:49:59+00:00
1,ciclovias_capa_01.parquet,Carril de prioridad ciclista,False,True,data/raw/urban_environment/ciclovias_semovi_20...,2026-10-11 23:49:59+00:00
2,ciclovias_capa_02.parquet,Ciclocarril,False,True,data/raw/urban_environment/ciclovias_semovi_20...,2026-10-11 23:49:59+00:00
3,ciclovias_capa_03.parquet,Ciclovia bidireccional,False,True,data/raw/urban_environment/ciclovias_semovi_20...,2026-10-11 23:49:59+00:00
4,ciclovias_capa_04.parquet,Ciclovia,False,True,data/raw/urban_environment/ciclovias_semovi_20...,2026-10-11 23:49:59+00:00
5,ciclovias_capa_05.parquet,Infraestructura ciclista emergente,False,True,data/raw/urban_environment/ciclovias_semovi_20...,2026-10-11 23:49:59+00:00
6,ciclovias_capa_06.parquet,Infraestructura ciclista total,True,True,data/raw/urban_environment/ciclovias_semovi_20...,2026-10-11 23:49:59+00:00
7,ciclovias_capa_07.parquet,Sendero compartido,False,True,data/raw/urban_environment/ciclovias_semovi_20...,2026-10-11 23:49:59+00:00


,input_rows,accepted_rows,quarantine_rows,superseded_duplicates,reason_occurrences,null_counts
accidentes_viales_2025,472,472,0,0,{},"{'fid': 0, 'fid_1': 0, 'id': 0, 'fenomeno': 0,..."
incidentes_c5_2025,2115080,0,2115080,12,"{'outside_study_year': 2115080, 'superseded_du...","{'folio': 0, 'fecha_creacion': 0, 'hora_creaci..."
ssc_victimas_2025,29706,29706,0,0,{},"{'lat': 0, 'lon': 0, 'category': 0, 'label': 0..."


,lat,lon,category,label,_source_file,_processed_at,_source_row,_raw_record,_record_sha256,_updated_at,_quarantine_reason,has_valid_coordinates,coordinates_zero_pair,ts_evento,is_possible_repeat
0,19.44776,-99.08476,Peatón Fallecido,SSC. Peatón fallecido 2025-01-02 0 HRS.,data/raw/urban_environment/ssc_semovi_2025_map...,2026-10-11 23:49:59+00:00,1,"{""category"":""Peatón Fallecido"",""label"":""SSC. P...",d5b3adfae5db99ca9c85eab7159f43059664bbb6115216...,NaT,,True,False,2025-01-02 00:00:00-06:00,False
1,19.377601,-99.06969,Peatón Fallecido,SSC. Peatón fallecido 2025-01-02 5 HRS.,data/raw/urban_environment/ssc_semovi_2025_map...,2026-10-11 23:49:59+00:00,2,"{""category"":""Peatón Fallecido"",""label"":""SSC. P...",32c506522b191b3a5860af2ed5b40132a8c48a807679e2...,NaT,,True,False,2025-01-02 05:00:00-06:00,False
2,19.385566,-99.037083,Peatón Fallecido,SSC. Peatón fallecido 2025-01-03 15 HRS.,data/raw/urban_environment/ssc_semovi_2025_map...,2026-10-11 23:49:59+00:00,3,"{""category"":""Peatón Fallecido"",""label"":""SSC. P...",220e303ef80e9be86b59273bbaf109ed38d85b5e0adf10...,NaT,,True,False,2025-01-03 15:00:00-06:00,False
3,19.301008,-99.161638,Peatón Fallecido,SSC. Peatón fallecido 2025-01-08 8 HRS.,data/raw/urban_environment/ssc_semovi_2025_map...,2026-10-11 23:49:59+00:00,4,"{""category"":""Peatón Fallecido"",""label"":""SSC. P...",0eb93ed99499c98de77610fa2231d87b87e5b68518ec52...,NaT,,True,False,2025-01-08 08:00:00-06:00,False
4,19.456721,-99.071057,Peatón Fallecido,SSC. Peatón fallecido 2025-01-11 6 HRS.,data/raw/urban_environment/ssc_semovi_2025_map...,2026-10-11 23:49:59+00:00,5,"{""category"":""Peatón Fallecido"",""label"":""SSC. P...",0c57ed70b1af41694cbeaebc426be37c75bc464443cfce...,NaT,,True,False,2025-01-11 06:00:00-06:00,False


**Resultados.** Publicamos capas y tablas con su partición de aceptación y cuarentena. Los indicadores de uso, vigencia y repetición se conservan para seleccionar fuentes y unidades de análisis en las etapas siguientes.

### Hallazgos temporales de C5 y límites del universo analítico

La fuente C5 contiene 2,115,080 registros con fechas entre el 31 de diciembre
de 2013 y el 29 de febrero de 2024. No contiene observaciones de 2025.
En consecuencia, la tabla `incidentes_c5_2025` queda vacía y conservamos las
filas fuera del período en cuarentena; doce también se identificaron como
duplicados desplazados. La ausencia de registros para 2025 no representa
ausencia de incidentes ni justifica asignar riesgo cero.

Esta fuente puede apoyar una revisión histórica con un universo temporal
distinto, pero no se incorpora como medición vial contemporánea de 2025.
Los registros fechados en 2025 de accidentes y víctimas SSC se mantienen
como fuentes separadas y no se suman entre sí sin comprobar solapamientos
y unidad de observación.

## Motivos de cuarentena de viajes

Una fila de viaje puede incumplir el contrato completo y conservar un extremo
válido. Por ello diferenciamos la cohorte de viajes de la actividad por eventos:
la llegada de enero de un viaje iniciado en diciembre de 2024 sigue contando
en 2025. Verificamos que cada fila pertenece a una sola partición y que el
peso de eventos válidos equivale a la suma de los pesos catalogados y no
catalogados.

El código `1000` no aparece en el catálogo disponible. Esa ausencia justifica
su tratamiento como identidad no resuelta; no permite afirmar que corresponda
a un taller, una estación de prueba o un servicio público. La cuarentena
conserva el motivo, el registro original y su origen para una revisión posterior.

In [8]:
dlq = pd.concat([pd.read_parquet(path, columns=["_quarantine_reason"])
                 for path in sorted(quarantine.glob("viajes_2025-*.parquet"))], ignore_index=True)
breakdown = dlq._quarantine_reason.str.split("|").explode().value_counts().rename("ocurrencias")
display(breakdown.to_frame())
quality = pd.read_parquet(processed / "auditoria_viajes.parquet")
assert quality.accepted_rows.add(quality.quarantine_rows).eq(quality.input_rows).all()
for role in ["retiros", "arribos"]:
    residual = quality["valid_" + role] - quality["known_" + role + "_weight"] - quality["unknown_" + role + "_weight"]
    assert residual.abs().lt(1e-5).all()
display(Markdown(f"**Resultados.** Conservamos {len(dlq):,} viajes en cuarentena. "
                 "Una fila puede tener varios motivos; la suma de ocurrencias no es el "
                 "número de filas rechazadas. La conservación de filas y pesos se verifica por archivo."))

,ocurrencias
_quarantine_reason,
arribos_station_outside_catalog,527
retiros_station_id_invalid,399
arribos_station_id_invalid,373
retiros_outside_2025,133
retiros_station_outside_catalog,66


**Resultados.** Conservamos 1,360 viajes en cuarentena. Una fila puede tener varios motivos; la suma de ocurrencias no es el número de filas rechazadas. La conservación de filas y pesos se verifica por archivo.

## Cierre, documentación y lectura de la generación activa

Verificamos los hashes de los artefactos y las fuentes al cerrar el proceso.
Los cuadernos posteriores resuelven `get_processed_root(V3)` en lugar de
escribir un hash fijo. El lector comprueba el manifiesto, el compromiso de
la generación, las rutas y la integridad de sus archivos sin modificar datos.

La preparación de fuentes no equivale a ausencia de faltantes ni a validación
de todas sus afirmaciones sustantivas. Los indicadores conservados delimitan
qué operaciones son posibles y qué supuestos adicionales requieren el EDA
y la construcción de atributos. Una nueva ejecución reutiliza únicamente
artefactos verificados bajo el mismo contrato.

In [9]:
display(pd.read_parquet(processed / "source_disposition.parquet"))
display(pd.read_parquet(processed / "raw_mirrors.parquet"))
display(pd.read_parquet(processed / "documentacion_fuentes.parquet")[["resource_type", "_source_file"]])
verify_outputs(V3, audit)
assert verify_manifest(V3) == manifest
assert run_etl(V3) == audit
display(Markdown("**Resultados finales.** La generación activa está verificada. "
                 "Todas las fuentes registradas tienen un destino de normalización, "
                 "documentación o comprobación de copia, sin eliminar RAW ni generaciones anteriores."))

,path,disposition,sha256,_source_file,_processed_at
0,data/raw/ecobici_stations/station_information....,normalized_core,8b4eaa98653beea3d51dc3d894ea8d5ee749923daee40d...,data/raw/ecobici_stations/station_information....,2026-10-11 23:49:59+00:00
1,data/raw/ecobici_trips/2025-01.csv,normalized_core,6b23c3d000856dabb72802eeb6cc84fdbbe0e5d1046019...,data/raw/ecobici_trips/2025-01.csv,2026-10-11 23:49:59+00:00
2,data/raw/ecobici_trips/2025-02.csv,normalized_core,f61d8be8138e1f6f922335b807626db6649d7936bc859a...,data/raw/ecobici_trips/2025-02.csv,2026-10-11 23:49:59+00:00
3,data/raw/ecobici_trips/2025-03.csv,normalized_core,a7cd8385505b596f895976bfb42d78dc4204c1b6202559...,data/raw/ecobici_trips/2025-03.csv,2026-10-11 23:49:59+00:00
4,data/raw/ecobici_trips/2025-04.csv,normalized_core,d4cb45a4280c486ce905bfa932b9562731356cb979112e...,data/raw/ecobici_trips/2025-04.csv,2026-10-11 23:49:59+00:00
5,data/raw/ecobici_trips/2025-05.csv,normalized_core,943aa9677b9440abd1a83fce52946c5909c6699f82e287...,data/raw/ecobici_trips/2025-05.csv,2026-10-11 23:49:59+00:00
6,data/raw/ecobici_trips/2025-06.csv,normalized_core,e1f6f5f07436aa03668811f1f7e0d00aeeff411eeafbb6...,data/raw/ecobici_trips/2025-06.csv,2026-10-11 23:49:59+00:00
7,data/raw/ecobici_trips/2025-07.csv,normalized_core,0a8bf13ce0b0bf1ba395435a07178bf7ac4381773bd63b...,data/raw/ecobici_trips/2025-07.csv,2026-10-11 23:49:59+00:00
8,data/raw/ecobici_trips/2025-08.csv,normalized_core,9be378627a1b4244a98d357c4d9c7a74570566b5598449...,data/raw/ecobici_trips/2025-08.csv,2026-10-11 23:49:59+00:00
9,data/raw/ecobici_trips/2025-09.csv,normalized_core,ea5bafb251c93265a759fc2ce3d43b136d4fa75938af52...,data/raw/ecobici_trips/2025-09.csv,2026-10-11 23:49:59+00:00


,path,archive,member,sha256,status,_source_file,_processed_at
0,data/raw/gtfs/gtfs/agency.txt,data/raw/gtfs/gtfs.zip,agency.txt,eb6d6fe12d9742f6510b2816f523bb0c6abbb9696da49f...,verified_not_double_ingested,data/raw/gtfs/gtfs/agency.txt,2026-10-11 23:49:59+00:00
1,data/raw/gtfs/gtfs/calendar.txt,data/raw/gtfs/gtfs.zip,calendar.txt,77221ae329a7e9f844134cb25a0cb81f9797b62cc8cdb8...,verified_not_double_ingested,data/raw/gtfs/gtfs/calendar.txt,2026-10-11 23:49:59+00:00
2,data/raw/gtfs/gtfs/frequencies.txt,data/raw/gtfs/gtfs.zip,frequencies.txt,b50d15c91199cf01fc906e4a537c9a06c07a7a477633d3...,verified_not_double_ingested,data/raw/gtfs/gtfs/frequencies.txt,2026-10-11 23:49:59+00:00
3,data/raw/gtfs/gtfs/routes.txt,data/raw/gtfs/gtfs.zip,routes.txt,cead5d1efb6ebec22224ae074f1211f0652276566f8c31...,verified_not_double_ingested,data/raw/gtfs/gtfs/routes.txt,2026-10-11 23:49:59+00:00
4,data/raw/gtfs/gtfs/shapes.txt,data/raw/gtfs/gtfs.zip,shapes.txt,a315b8e4adc737ed1a29c36fd500704fae882184329f31...,verified_not_double_ingested,data/raw/gtfs/gtfs/shapes.txt,2026-10-11 23:49:59+00:00
5,data/raw/gtfs/gtfs/stop_times.txt,data/raw/gtfs/gtfs.zip,stop_times.txt,f121a29ef7a86cedf5738bdf018b8d1d39cdf8e31b13f4...,verified_not_double_ingested,data/raw/gtfs/gtfs/stop_times.txt,2026-10-11 23:49:59+00:00
6,data/raw/gtfs/gtfs/stops.txt,data/raw/gtfs/gtfs.zip,stops.txt,f1aa8c5b33f1b04bfd40a8d02564c174737a6e8efbadd6...,verified_not_double_ingested,data/raw/gtfs/gtfs/stops.txt,2026-10-11 23:49:59+00:00
7,data/raw/gtfs/gtfs/trips.txt,data/raw/gtfs/gtfs.zip,trips.txt,562f809db1cb62a9ecf44ab42999c3b1e0de362de87a28...,verified_not_double_ingested,data/raw/gtfs/gtfs/trips.txt,2026-10-11 23:49:59+00:00
8,data/raw/inegi_ageb/denue_09_csv/conjunto_de_d...,data/raw/inegi_ageb/denue_09_csv.zip,conjunto_de_datos/denue_inegi_09_.csv,e12fcc6c4d518a169c9445c7f160e9d6fc9ba02fd0a84d...,verified_not_double_ingested,data/raw/inegi_ageb/denue_09_csv/conjunto_de_d...,2026-10-11 23:49:59+00:00
9,data/raw/inegi_ageb/denue_09_csv/diccionario_d...,data/raw/inegi_ageb/denue_09_csv.zip,diccionario_de_datos/denue_diccionario_de_dato...,53619afc3424522a2bb1587adc34b45b9660728df2d2f4...,verified_not_double_ingested,data/raw/inegi_ageb/denue_09_csv/diccionario_d...,2026-10-11 23:49:59+00:00


,resource_type,_source_file
0,metadata,data/raw/inegi_ageb/denue_09_csv.zip::metadato...
1,metadata,data/raw/inegi_ageb/ageb_mza_urbana_09_cpv2020...
2,publication,data/raw/urban_environment/ssc_semovi_2025_pub...
3,source_registry,data/raw/urban_environment/fuentes_diagnostico...


**Resultados finales.** La generación activa está verificada. Todas las fuentes registradas tienen un destino de normalización, documentación o comprobación de copia, sin eliminar RAW ni generaciones anteriores.